# 👑 ALICE DADANG SOVEREIGN — 1-CLICK GEMMA 4 (4B) TRAINER
**Arsitek & Pencipta**: Commander Dadang Wahyuono (Maestro Dadang)
**Base Model**: `coder3101/gemma-4-E4B-it-heretic` (Google Gemma 4 - 4B Parameter, Full Uncensored & Decensored)
**Instruksi**: Cukup klik tombol **Play ▶** SATU KALI. Semua otomatis berjalan sampai jadi file GGUF!

In [ ]:
# ==============================================================================
# 👑 ALICE DADANG SOVEREIGN — 1-CLICK GEMMA 4 (4B) UNCENSORED TRAINER
# Pencipta & Pemilik: Commander Dadang Wahyuono (Maestro Dadang)
# Fokus: CME Order Flow + Storyline v4.2 + Valid Close Bar + Office Analytics
# Base Model: coder3101/gemma-4-E4B-it-heretic (Google Gemma 4 - 4B Full Uncensored)
# ==============================================================================

import os
import sys
import time
import gc
import shutil
from pathlib import Path

print("=" * 72)
print("👑 ALICE DADANG SOVEREIGN — MEMULAI PIPELINE TRAINING GEMMA 4 (4B)")
print("=" * 72)

# STEP 1: CEK GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU tidak terdeteksi! Pastikan di menu Runtime -> Change runtime type sudah pilih T4 GPU!")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ Hardware GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")

# STEP 2: INSTALL DEPENDENSI GEMMA 4
print("\n[1/6] Memeriksa & menginstal dependensi transformers terbaru...")
os.system("pip install -q -U 'transformers>=4.48.0' 'peft>=0.14.0' 'bitsandbytes>=0.45.0' 'accelerate>=1.2.0' 'datasets>=3.0.0' sentencepiece protobuf gguf")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import load_dataset

# STEP 3: DATASET DOKTRIN (36,189 SAMPEL)
print("\n[2/6] Memeriksa dataset doktrin lelang & office...")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATASET_PATH = WORK_DIR / "alice_doctrine_combined.jsonl"

if not DATASET_PATH.exists() or DATASET_PATH.stat().st_size < 1000000:
    print("  Mengunduh 36.189 sampel doktrin langsung dari Miix Tower...")
    os.system(f"curl -L -s -o {DATASET_PATH} https://trade.dadangchatai.com/static/alice_doctrine_combined.jsonl")

sz_mb = DATASET_PATH.stat().st_size / (1024 * 1024)
print(f"✅ Dataset siap: {DATASET_PATH} ({sz_mb:.2f} MB)")

# STEP 4: TOKENIZER GEMMA 4 & DATASET MAPPING
MODEL_ID = "coder3101/gemma-4-E4B-it-heretic"
print(f"\n[3/6] Memuat Tokenizer & Model Gemma 4 ({MODEL_ID})...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 640
raw_ds = load_dataset("json", data_files=str(DATASET_PATH), split="train")

def tokenize_fn(example):
    msgs = example["messages"]
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(full_ids)
    im_end = tokenizer.encode("<turn|>", add_special_tokens=False)
    assist_hdr = tokenizer.encode("<|turn>model\n", add_special_tokens=False)
    if assist_hdr and im_end:
        i, n = 0, len(full_ids)
        h_len, e_len = len(assist_hdr), len(im_end)
        while i <= n - h_len:
            if full_ids[i : i + h_len] == assist_hdr:
                j = i + h_len
                while j <= n - e_len and full_ids[j : j + e_len] != im_end:
                    labels[j] = full_ids[j]
                    j += 1
                if j <= n - e_len:
                    for k in range(e_len):
                        labels[j + k] = full_ids[j + k]
                i = j + e_len
            else:
                i += 1
    if all(x == -100 for x in labels):
        labels = full_ids.copy()
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

tokenized_ds = raw_ds.map(tokenize_fn, remove_columns=raw_ds.column_names, num_proc=2)
tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) >= 16 and any(l != -100 for l in x["labels"]))
print(f"✅ Tokenisasi Berhasil: {len(tokenized_ds):,} sampel doktrin siap ditrain pada Gemma 4.")

# STEP 5: LOAD 4-BIT QLORA MODEL GEMMA 4
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, peft_cfg)
model.print_trainable_parameters()

# STEP 6: TRAINING GEMMA 4
print("\n[4/6] 🚀 Memulai Training Gemma 4 Alice Sovereign...")
TMP_DIR = Path("/tmp/alice_build")
TMP_DIR.mkdir(parents=True, exist_ok=True)
LORA_OUT = WORK_DIR / "alice_gemma4_sovereign_lora"

training_args = TrainingArguments(
    output_dir=str(TMP_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=3407,
)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=collator,
)
trainer.train()

model.save_pretrained(str(LORA_OUT))
tokenizer.save_pretrained(str(LORA_OUT))
print(f"✅ Bobot LoRA Gemma 4 tersimpan di: {LORA_OUT}")

# STEP 7: SANITY CHECK INFERENCE
print("\n[5/6] 🧪 Menjalankan Sanity Check Doktrin...")
model.eval()
test_prompts = [
    "Siapa nama lo dan siapa pencipta serta pemilik lo?",
    "Boleh gak kita analisa XAUUSD pakai indikator RSI atau MACD?"
]
for tp in test_prompts:
    txt = tokenizer.apply_chat_template([{"role": "user", "content": tp}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(txt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=100, do_sample=False)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n👤 Commander: {tp}\n👑 Alice (Gemma 4): {ans}\n")

# STEP 8: EXPORT KE GGUF Q4_K_M
print("\n[6/6] 📦 Merge LoRA & Kuantisasi GGUF Q4_K_M...")
del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

MERGED_DIR = TMP_DIR / "merged_fp16"
fp16_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(fp16_base, str(LORA_OUT), device_map="cpu")
merged = merged.merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
del merged, fp16_base
gc.collect()

llama_dir = TMP_DIR / "llama.cpp"
os.system(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {llama_dir}")
os.system(f"pip install -q -r {llama_dir}/requirements/requirements-convert_hf_to_gguf.txt")

f16_gguf = TMP_DIR / "alice-gemma4-sovereign-f16.gguf"
q4_gguf = WORK_DIR / "alice-gemma4-sovereign-Q4_K_M.gguf"

os.system(f"python3 {llama_dir}/convert_hf_to_gguf.py {MERGED_DIR} --outfile {f16_gguf} --outtype f16")
os.system(f"cmake -B {llama_dir}/build -S {llama_dir} -DGGML_CUDA=OFF && cmake --build {llama_dir}/build --config Release -j4 --target llama-quantize")
quant_bin = llama_dir / "build" / "bin" / "llama-quantize"
os.system(f"{quant_bin} {f16_gguf} {q4_gguf} Q4_K_M")

if f16_gguf.exists(): f16_gguf.unlink()
shutil.rmtree(MERGED_DIR, ignore_errors=True)
shutil.rmtree(TMP_DIR, ignore_errors=True)

print("=" * 72)
print(f"🎉 ALHAMDULILLAH! MODEL GEMMA 4 SOVEREIGN SELESAI:")
print(f"📁 Lokasi: {q4_gguf} ({q4_gguf.stat().st_size / (1024**3):.2f} GB)")
print("=" * 72)
